# 09 — Test-set inference for every saved run

For each `runs/<run_id>/result.json`, re-evaluate the same config on the **test** set
(originally the ImageNet validation split at `/home/pf4636/imagenet2/val`) and write a
fresh result JSON to `runs/final_runs/<run_id>/`.

- pytorch backend → load weights → eval on test loader
- torchao_cpu_ptq → re-calibrate on train loader → eval on test loader
- tensorrt → reuse existing engine in `engines/` → trt_evaluate on test loader

In [1]:
import sys, json, time
from pathlib import Path
from dataclasses import replace

SRC = Path("..").resolve() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from config import ExperimentConfig, set_seed
from data import build_imagenet_dataset, build_runner_loaders
from model import get_model
from precision import apply_precision
from quant_ptq_cpu import quantize_int8_x86_pt2e
from eval import evaluate
from utils import iter_result_jsons, read_json, write_json
from runner import _get_trt_paths

TEST_ROOT  = "/home/pf4636/imagenet2"
RUNS_IN    = Path("../runs/val_infer").resolve()
RUNS_OUT   = Path("../runs/final_runs").resolve()
RUNS_OUT.mkdir(parents=True, exist_ok=True)
print(f"reading from: {RUNS_IN}")
print(f"writing to  : {RUNS_OUT}")

reading from: /home/pf4636/code/resnet/quantized_resnets/runs/val_infer
writing to  : /home/pf4636/code/resnet/quantized_resnets/runs/final_runs


In [2]:
def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_ROOT)
    dataset = build_imagenet_dataset(test_cfg, split="val")
    pin_memory = str(cfg.device).startswith("cuda")
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
        drop_last=True,
    )

def run_on_test(cfg: ExperimentConfig, criterion=None):
    cfg = cfg.normalized()
    cfg.validate()
    set_seed(cfg)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    test_loader = build_test_loader(cfg)
    t0 = time.perf_counter()

    if cfg.backend == "pytorch":
        model = apply_precision(get_model(cfg), cfg)
        tracker = evaluate(model, test_loader, cfg, criterion=criterion)

    elif cfg.backend == "torchao_cpu_ptq":
        train_loader, _ = build_runner_loaders(cfg)
        model = quantize_int8_x86_pt2e(
            get_model(cfg), train_loader,
            calib_num_batches=cfg.cpu_calib_num_batches,
        )
        tracker = evaluate(model, test_loader, cfg, criterion=criterion)

    elif cfg.backend == "tensorrt":
        from trt_infer import trt_evaluate
        from onnx_exporter import ONNXExporter
        from trt_builder import build_engine

        onnx_path, engine_path, _ = _get_trt_paths(cfg)
        if not onnx_path.exists():
            ONNXExporter(get_model(cfg), cfg.device, onnx_path).export_model(
                opset_version=cfg.trt_opset if cfg.trt_opset > 1 else 17,
                dynamic_batch=True,
                dummy_input_shape=(1, 3, 224, 224),
            )
        if not engine_path.exists():
            build_engine(
                onnx_path=onnx_path,
                engine_path=engine_path,
                precision=cfg.model_precision,
                batch_size=cfg.batch_size,
                workspace_mb=cfg.trt_workspace_mb,
            )
        tracker = trt_evaluate(engine_path, cfg, test_loader, criterion)

    else:
        raise ValueError(f"Unknown backend: {cfg.backend}")

    payload = {
        "status": "ok",
        "run_id": cfg.run_id(),
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "results": tracker.summary(),
        "error": None,
        "total_eval_time_sec": round(time.perf_counter() - t0, 3),
    }
    return payload, tracker

In [3]:
def cfg_from_payload(payload: dict) -> ExperimentConfig:
    raw = payload["config"]
    fields = {f for f in ExperimentConfig.__dataclass_fields__}
    return ExperimentConfig(**{k: v for k, v in raw.items() if k in fields})

configs = []
for p in iter_result_jsons(RUNS_IN):
    if RUNS_OUT in p.parents:
        continue
    payload = read_json(p)
    if payload.get("status") != "ok":
        continue
    configs.append(cfg_from_payload(payload))

print(f"found {len(configs)} runs to re-evaluate on test set")
for c in configs:
    print("  ", c.run_id())

found 32 runs to re-evaluate on test set
   resnet18_pytorch_fp16_in1b_cuda_bs1
   resnet18_pytorch_fp16_in2b_cuda_bs1
   resnet18_pytorch_fp16_in4b_cuda_bs1
   resnet18_pytorch_fp16_in8b_cuda_bs1
   resnet18_pytorch_fp32_in1b_cuda_bs1
   resnet18_pytorch_fp32_in2b_cuda_bs1
   resnet18_pytorch_fp32_in4b_cuda_bs1
   resnet18_pytorch_fp32_in8b_cuda_bs1
   resnet18_tensorrt_fp16_in1b_cuda_bs1
   resnet18_tensorrt_fp16_in2b_cuda_bs1
   resnet18_tensorrt_fp16_in4b_cuda_bs1
   resnet18_tensorrt_fp16_in8b_cuda_bs1
   resnet18_tensorrt_fp32_in1b_cuda_bs1
   resnet18_tensorrt_fp32_in2b_cuda_bs1
   resnet18_tensorrt_fp32_in4b_cuda_bs1
   resnet18_tensorrt_fp32_in8b_cuda_bs1
   resnet18_tensorrt_fp8_in1b_cuda_bs1
   resnet18_tensorrt_fp8_in2b_cuda_bs1
   resnet18_tensorrt_fp8_in4b_cuda_bs1
   resnet18_tensorrt_fp8_in8b_cuda_bs1
   resnet18_tensorrt_int4_in1b_cuda_bs1
   resnet18_tensorrt_int4_in2b_cuda_bs1
   resnet18_tensorrt_int4_in4b_cuda_bs1
   resnet18_tensorrt_int4_in8b_cuda_bs1
   resnet18

In [4]:
SKIP_EXISTING = True

results_summary = []
for i, cfg in enumerate(configs, 1):
    out_path = RUNS_OUT / cfg.run_id() / "result.json"
    print(f"\n[{i}/{len(configs)}] {cfg.run_id()}")

    if SKIP_EXISTING and out_path.exists():
        print(f"  already exists, skipping: {out_path}")
        results_summary.append(read_json(out_path))
        continue

    try:
        payload, _ = run_on_test(cfg)
        write_json(out_path, payload)
        print(f"  saved: {out_path}")
        r = payload["results"]
        print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f} ms")
        results_summary.append(payload)
    except Exception as e:
        err_payload = {
            "status": "error",
            "run_id": cfg.run_id(),
            "config": cfg.to_dict(),
            "error": f"{type(e).__name__}: {e}",
        }
        write_json(out_path, err_payload)
        print(f"  FAILED: {e}")
        results_summary.append(err_payload)


[1/32] resnet18_pytorch_fp16_in1b_cuda_bs1
[data] Filtered to 5000 samples across 100 classes.
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 20.00% | Top-5: 60.00% | Infer: 2.91 ms/batch
  Batch [50/500] Top-1: 40.00% | Top-5: 75.00% | Infer: 2.77 ms/batch
  Batch [60/500] Top-1: 33.33% | Top-5: 80.00% | Infer: 2.89 ms/batch
  Batch [70/500] Top-1: 37.50% | Top-5: 82.50% | Infer: 2.80 ms/batch
  Batch [80/500] Top-1: 42.00% | Top-5: 82.00% | Infer: 2.85 ms/batch
  Batch [90/500] Top-1: 46.67% | Top-5: 85.00% | Infer: 2.78 ms/batch
  Batch [100/500] Top-1: 50.00% | Top-5: 87.14% | Infer: 2.72 ms/batch
  Batch [110/500] Top-1: 46.25% | Top-5: 80.00% | Infer: 2.83 ms/batch
  Batch [120/500] Top-1: 42.22% | Top-5: 75.56% | Infer: 2.90 ms/batch
  Batch [130/500] Top-1: 38.00% | Top-5: 70.00% | Infer: 2.90 ms/batch
  Batch [140/500] Top-1: 35.45% | Top-5: 70.00% | Infer: 2.87 ms/batch
  Batch [1

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/torchao/quantization/pt2e/quantizer/x86_inductor_quantizer.py:1325: UserWarning: The input of maxpool2d is not quantized, skip annotate maxpool2d with config QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), weight=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.PerChannelMinMaxObserver'>, eps=0.000244140625){}, quant

Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 20.00% | Top-5: 60.00% | Infer: 11.07 ms/batch
  Batch [50/500] Top-1: 40.00% | Top-5: 75.00% | Infer: 10.69 ms/batch
  Batch [60/500] Top-1: 33.33% | Top-5: 80.00% | Infer: 12.41 ms/batch
  Batch [70/500] Top-1: 37.50% | Top-5: 82.50% | Infer: 12.34 ms/batch
  Batch [80/500] Top-1: 42.00% | Top-5: 82.00% | Infer: 12.22 ms/batch
  Batch [90/500] Top-1: 46.67% | Top-5: 85.00% | Infer: 11.96 ms/batch
  Batch [100/500] Top-1: 50.00% | Top-5: 87.14% | Infer: 12.34 ms/batch
  Batch [110/500] Top-1: 46.25% | Top-5: 80.00% | Infer: 12.36 ms/batch
  Batch [120/500] Top-1: 42.22% | Top-5: 75.56% | Infer: 12.29 ms/batch
  Batch [130/500] Top-1: 38.00% | Top-5: 70.00% | Infer: 12.15 ms/batch
  Batch [140/500] Top-1: 35.45% | Top-5: 70.00% | Infer: 12.66 ms/batch
  Batch [150/500] Top-1: 35.00% | Top-5: 70.00% | Infer: 12.75 ms/batch
  Batch [160/500] Top-1

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/torchao/quantization/pt2e/quantizer/x86_inductor_quantizer.py:1325: UserWarning: The input of maxpool2d is not quantized, skip annotate maxpool2d with config QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), weight=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.PerChannelMinMaxObserver'>, eps=0.000244140625){}, quant

Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 80.00% | Top-5: 80.00% | Infer: 10.35 ms/batch
  Batch [50/500] Top-1: 85.00% | Top-5: 85.00% | Infer: 11.86 ms/batch
  Batch [60/500] Top-1: 83.33% | Top-5: 90.00% | Infer: 11.04 ms/batch
  Batch [70/500] Top-1: 77.50% | Top-5: 90.00% | Infer: 10.49 ms/batch
  Batch [80/500] Top-1: 78.00% | Top-5: 88.00% | Infer: 10.36 ms/batch
  Batch [90/500] Top-1: 76.67% | Top-5: 90.00% | Infer: 10.32 ms/batch
  Batch [100/500] Top-1: 75.71% | Top-5: 90.00% | Infer: 10.35 ms/batch
  Batch [110/500] Top-1: 71.25% | Top-5: 88.75% | Infer: 10.66 ms/batch
  Batch [120/500] Top-1: 67.78% | Top-5: 88.89% | Infer: 10.49 ms/batch
  Batch [130/500] Top-1: 65.00% | Top-5: 86.00% | Infer: 10.55 ms/batch
  Batch [140/500] Top-1: 62.73% | Top-5: 86.36% | Infer: 11.33 ms/batch
  Batch [150/500] Top-1: 60.00% | Top-5: 83.33% | Infer: 11.30 ms/batch
  Batch [160/500] Top-1

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/torchao/quantization/pt2e/quantizer/x86_inductor_quantizer.py:1325: UserWarning: The input of maxpool2d is not quantized, skip annotate maxpool2d with config QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), weight=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.PerChannelMinMaxObserver'>, eps=0.000244140625){}, quant

Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 80.00% | Top-5: 100.00% | Infer: 10.09 ms/batch
  Batch [50/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 9.86 ms/batch
  Batch [60/500] Top-1: 93.33% | Top-5: 100.00% | Infer: 9.88 ms/batch
  Batch [70/500] Top-1: 92.50% | Top-5: 100.00% | Infer: 9.78 ms/batch
  Batch [80/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.62 ms/batch
  Batch [90/500] Top-1: 91.67% | Top-5: 100.00% | Infer: 10.64 ms/batch
  Batch [100/500] Top-1: 91.43% | Top-5: 100.00% | Infer: 10.85 ms/batch
  Batch [110/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.92 ms/batch
  Batch [120/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.85 ms/batch
  Batch [130/500] Top-1: 91.00% | Top-5: 100.00% | Infer: 10.83 ms/batch
  Batch [140/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 11.28 ms/batch
  Batch [150/500] Top-1: 87.50% | Top-5: 100.00% | Infer: 11.39 ms/batch
  Batch [160/5

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/torchao/quantization/pt2e/quantizer/x86_inductor_quantizer.py:1325: UserWarning: The input of maxpool2d is not quantized, skip annotate maxpool2d with config QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), weight=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.PerChannelMinMaxObserver'>, eps=0.000244140625){}, quant

Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.27 ms/batch
  Batch [50/500] Top-1: 95.00% | Top-5: 100.00% | Infer: 10.72 ms/batch
  Batch [60/500] Top-1: 96.67% | Top-5: 100.00% | Infer: 10.42 ms/batch
  Batch [70/500] Top-1: 92.50% | Top-5: 100.00% | Infer: 10.30 ms/batch
  Batch [80/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.29 ms/batch
  Batch [90/500] Top-1: 91.67% | Top-5: 100.00% | Infer: 10.36 ms/batch
  Batch [100/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 12.33 ms/batch
  Batch [110/500] Top-1: 88.75% | Top-5: 100.00% | Infer: 12.20 ms/batch
  Batch [120/500] Top-1: 87.78% | Top-5: 100.00% | Infer: 12.07 ms/batch
  Batch [130/500] Top-1: 89.00% | Top-5: 100.00% | Infer: 12.04 ms/batch
  Batch [140/500] Top-1: 89.09% | Top-5: 100.00% | Infer: 12.50 ms/batch
  Batch [150/500] Top-1: 87.50% | Top-5: 100.00% | Infer: 12.25 ms/batch
  Batch [16

In [5]:
import pandas as pd
from utils import flatten_runs, load_runs

test_runs = load_runs(RUNS_OUT, status="ok")
df = pd.DataFrame(flatten_runs(test_runs))
cols = ["run_id", "cfg.backend", "cfg.model_precision", "cfg.input_quant_bits",
        "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.infer_ms_std"]
df[[c for c in cols if c in df.columns]].sort_values(
    ["cfg.backend", "cfg.input_quant_bits", "cfg.model_precision"]
).reset_index(drop=True)

,run_id,cfg.backend,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.infer_ms_std
0,resnet18_pytorch_fp16_in1b_cuda_bs1,pytorch,fp16,1,31.914894,63.617021,3.041370,1.503594
1,resnet18_pytorch_fp32_in1b_cuda_bs1,pytorch,fp32,1,31.489362,63.617021,3.053116,1.303294
2,resnet18_pytorch_fp16_in2b_cuda_bs1,pytorch,fp16,2,51.914894,81.063830,2.943886,1.567814
3,resnet18_pytorch_fp32_in2b_cuda_bs1,pytorch,fp32,2,51.914894,81.063830,2.840510,1.230063
4,resnet18_pytorch_fp16_in4b_cuda_bs1,pytorch,fp16,4,80.851064,96.595745,2.955571,1.482245
5,resnet18_pytorch_fp32_in4b_cuda_bs1,pytorch,fp32,4,80.851064,96.595745,2.713301,1.138876
6,resnet18_pytorch_fp16_in8b_cuda_bs1,pytorch,fp16,8,80.638298,97.234043,2.909420,1.458176
7,resnet18_pytorch_fp32_in8b_cuda_bs1,pytorch,fp32,8,80.638298,97.234043,2.911906,1.322866
8,resnet18_tensorrt_fp16_in1b_cuda_bs1,tensorrt,fp16,1,31.914894,63.404255,0.463038,0.171537
9,resnet18_tensorrt_fp32_in1b_cuda_bs1,tensorrt,fp32,1,31.702128,63.617021,0.931417,0.129937
